# ch04 Final (clean)
A cleaned, well-formatted copy of chapter 4 with GPU-aware device handling.

In [ ]:
# Imports and device selection
import sys
import tiktoken
import torch
import torch.nn as nn
# select CUDA when available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
print("sys.executable:", sys.executable)

In [ ]:
# Model configuration
GPT_CONFIG_124M = {
    "vocab_size": 50257,
    "context_length": 1024,
    "emb_dim": 768,
    "n_heads": 12,
    "n_layers": 12,
    "drop_rate": 0.1,
    "qkv_bias": False
}

In [ ]:
class LayerNorm(nn.Module):
    def __init__(self, embed_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(embed_dim))
        self.shift = nn.Parameter(torch.zeros(embed_dim))

    def forward(self, x):
        local_mean = x.mean(dim=-1, keepdim=True)
        local_var = x.var(dim=-1, keepdim=True)
        norm_x = (x - local_mean) / torch.sqrt(local_var + self.eps)
        return norm_x * self.scale + self.shift

In [ ]:
class GELU(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(
            torch.sqrt(torch.tensor(2.0 / torch.pi)) * (x + 0.044715 * torch.pow(x, 3))
        ))

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(emb_dim, 4 * emb_dim),
            GELU(),
            nn.Linear(4 * emb_dim, emb_dim),
        )

    def forward(self, x):
        return self.layers(x)

In [ ]:
def causal_mask(seq_len, device=None):
    mask = torch.triu(torch.ones(seq_len, seq_len, device=device), diagonal=1).bool()
    return mask

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, num_heads, dropout=0.0):
        super().__init__()
        assert d_out % num_heads == 0, 'd_out must be divisible by num_heads'
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads
        self.scale = self.head_dim ** 0.5
        self.W_query = nn.Linear(d_in, d_out, bias=False)
        self.W_key = nn.Linear(d_in, d_out, bias=False)
        self.W_value = nn.Linear(d_in, d_out, bias=False)
        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        batch, seq_len, _ = x.shape
        query = self.W_query(x).view(batch, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        key = self.W_key(x).view(batch, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        value = self.W_value(x).view(batch, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        attn_scores = query @ key.transpose(-2, -1)
        mask = causal_mask(seq_len, device=x.device)
        attn_scores = attn_scores.masked_fill(mask.unsqueeze(0).unsqueeze(0), float('-inf'))
        attn_weights = torch.softmax(attn_scores / self.scale, dim=-1)
        attn_weights = self.dropout(attn_weights)
        context = attn_weights @ value
        context = context.transpose(1, 2).contiguous().view(batch, seq_len, -1)
        return self.out_proj(context)

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.LayerNorm1 = LayerNorm(cfg['emb_dim'])
        self.LayerNorm2 = LayerNorm(cfg['emb_dim'])
        self.MultiHAttn = MultiHeadAttention(cfg['emb_dim'], cfg['emb_dim'], cfg['n_heads'], cfg['drop_rate'])
        self.Dropout1 = nn.Dropout(cfg['drop_rate'])
        self.Dropout2 = nn.Dropout(cfg['drop_rate'])
        self.FeedForward = FeedForward(cfg['emb_dim'])

    def forward(self, x):
        skp = x
        x = self.LayerNorm1(x)
        x = self.MultiHAttn(x)
        x = self.Dropout1(x)
        x = x + skp
        skp = x
        x = self.LayerNorm2(x)
        x = self.FeedForward(x)
        x = self.Dropout2(x)
        return x + skp

In [ ]:
class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg['vocab_size'], cfg['emb_dim'])
        self.pos_emb = nn.Embedding(cfg['context_length'], cfg['emb_dim'])
        self.drop_emb = nn.Dropout(cfg['drop_rate'])
        self.GPT_Block = nn.Sequential(*[TransformerBlock(cfg) for _ in range(cfg['n_layers'])])
        self.final_norm = LayerNorm(cfg['emb_dim'])
        self.out_head = nn.Linear(cfg['emb_dim'], cfg['vocab_size'], bias=False)

    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        tok_emb = self.tok_emb(in_idx)
        pos_emb = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        x = tok_emb + pos_emb
        x = self.drop_emb(x)
        x = self.GPT_Block(x)
        x = self.final_norm(x)
        logits = self.out_head(x)
        return logits

In [ ]:
# Tokenization and batch creation (on device)
tokenizer = tiktoken.get_encoding("gpt2")
txt1 = "Every effort moves you"
txt2 = "Every day holds a"
batch = []
batch.append(torch.tensor(tokenizer.encode(txt1), dtype=torch.long, device=device))
batch.append(torch.tensor(tokenizer.encode(txt2), dtype=torch.long, device=device))
batch = torch.stack(batch, dim=0)
print("Batch shape:", batch.shape)

In [ ]:
# Instantiate model and move to device
gptmodel = GPTModel(GPT_CONFIG_124M).to(device)
trainable_params = sum(p.numel() for p in gptmodel.parameters() if p.requires_grad)
print(f"Trainable parameters: {trainable_params:,}")

In [ ]:
def generate_text_simple(model, idx, max_new_tokens, context_size, device=device):
    if not isinstance(idx, torch.Tensor):
        idx = torch.tensor(idx, dtype=torch.long, device=device)
    else:
        idx = idx.to(device)
    model = model.to(device)
    model.eval()
    with torch.no_grad():
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -context_size:]
            logits = model(idx_cond)
            logits = logits[:, -1, :]
            probas = torch.softmax(logits, dim=-1)
            idx_next = torch.argmax(probas, dim=-1, keepdim=True)
            idx = torch.cat((idx, idx_next), dim=1)
    return idx

In [ ]:
# Small generation demo (move output to CPU then decode)
start_context = "Hello, I am "
encoded = tokenizer.encode(start_context)
encoded_tensor = torch.tensor(encoded, dtype=torch.long, device=device).unsqueeze(0)
out = generate_text_simple(gptmodel, encoded_tensor, max_new_tokens=50, context_size=GPT_CONFIG_124M['context_length'])
decoded_text = tokenizer.decode(out.squeeze(0).to('cpu').tolist())
print(decoded_text)